# 반도체 공정 스케줄링 시뮬레이션 결과

## 환경 설정

In [1]:
import os
import simpy
from dotenv import load_dotenv

# .env 파일의 환경 변수 로드
load_dotenv()

# .env 파일의 파라미터
SIMUL_TIME = int(os.getenv('SIMUL_TIME', 100))
BASE_DATA_PATH = os.getenv('BASE_DATA_PATH', 'data')

print(f"시뮬레이션 시간: {SIMUL_TIME}")
print(f"데이터 경로: {BASE_DATA_PATH}")

시뮬레이션 시간: 150
데이터 경로: data

## 모듈 import

In [2]:
from simulation import DataLoader, Scheduler, Job
from simulation.visualizer import create_job_gantt_chart
# 향후 알고리즘 모듈 추가 예정
# from algorithms.genetic import GeneticScheduler
# from algorithms.rule_based import RuleBasedScheduler

## 데이터 로드

In [3]:
data_loader = DataLoader(BASE_DATA_PATH)

data = data_loader.load_all_data()

print("=" * 60)
print("데이터 개요")
print("=" * 60)
print(f"Jobs: {len(data['jobs'])} 개")
print(f"Operations: {len(data['operations'])} 개")
print(f"Machines: {len(data['machines'])} 개")
print(f"Machine Failures: {len(data['machine_failure'])} 개")
print(f"Setup Times: {len(data['setup_times'])} 개")
print(f"Operation-Machine Map: {len(data['operation_machine_map'])} 개")

데이터 개요
Jobs: 10 개
Operations: 35 개
Machines: 8 개
Machine Failures: 8 개
Setup Times: 12 개
Operation-Machine Map: 95 개

## 시뮬레이션 실행

In [4]:
# SimPy 환경 생성
env = simpy.Environment()

# 스케줄러 생성
scheduler = Scheduler(
    env=env,
    machine_df=data['machines'],
    operations_df=data['operations'],
    machine_failure_df=data['machine_failure'],
    setup_times_df=data['setup_times'],
    op_machine_df=data['operation_machine_map']
)

## 작업 생성

In [5]:
jobs = []

for _, job_info in data['jobs'].iterrows():
    # 해당 작업의 operation 정보 가져오기
    job_operations = data['operations'].loc[
        data['operations']['job_id'] == job_info['job_id'],
        ['op_id', 'op_seq', 'qtime']
    ].sort_values('op_seq')
    
    # Job 인스턴스 생성
    job = Job(
        env=env,
        job_info=job_info.to_dict(),
        op_info=job_operations,
        scheduler=scheduler
    )
    jobs.append(job)

## 시뮬레이션 실행

In [6]:
env.run(until=SIMUL_TIME)
fig = create_job_gantt_chart(
    jobs=jobs,
    max_time=SIMUL_TIME,
    title=f"Job Gantt Chart (Simulation Time: {SIMUL_TIME})"
)
fig.show()

23.9    Machine M6 broke down
25.24   Machine M7 broke down
31.9    Machine M6 repaired
33.24   Machine M7 repaired
35.41   Machine M8 broke down
39.61   Machine M2 broke down
43.41   Machine M8 repaired
49.02   Machine M1 broke down
49.61   Machine M2 repaired
49.81   Machine M7 broke down
57.81   Machine M7 repaired
59.02   Machine M1 repaired
61.58   Machine M2 broke down
64.79   Machine M4 broke down
71.58   Machine M2 repaired
77.54   Machine M8 broke down
79.42   Machine M2 broke down
79.79   Machine M4 repaired
85.54   Machine M8 repaired
89.42   Machine M2 repaired
94.87   Machine M7 broke down
98.47   Machine M3 broke down
101.14  Machine M4 broke down
102.87  Machine M7 repaired
110.47  Machine M3 repaired
116.14  Machine M4 repaired
117.44  Machine M8 broke down
119.5   Machine M5 broke down
121.43  Machine M2 broke down
122.88  Machine M6 broke down
125.4   Machine M4 broke down
125.44  Machine M8 repaired
127.09  Machine M3 broke down
130.88  Machine M6 repaired
131.43  Ma

## 시뮬레이션 KPI 및 통계

In [7]:
completed_cnt = 0
total_qtime_violation = 0.0
for job in jobs:
    print(f"Job ID: {job.id}, QTime Violation: {round(job.calculate_qtime_over(SIMUL_TIME), 3)}")
    completed_cnt += int(job.is_completed)
    total_qtime_violation += job.total_qtime_over
print(f"총 작업 수: {len(jobs)}, 완료된 작업 수: {completed_cnt}, 총 QTime 위반 시간: {round(total_qtime_violation, 3)}")
# 향후 결과 분석 기능 추가
# - 작업 완료율
# - 평균 대기 시간
# - 장비 가동률
# - QTime 위반 건수
# - 처리량 (Throughput)

Job ID: J01, QTime Violation: 5.0
Job ID: J02, QTime Violation: 0.0
Job ID: J03, QTime Violation: 0.0
Job ID: J04, QTime Violation: 3.0
Job ID: J05, QTime Violation: 0.0
Job ID: J06, QTime Violation: 6.0
Job ID: J07, QTime Violation: 15.0
Job ID: J08, QTime Violation: 0.0
Job ID: J09, QTime Violation: 11.786
Job ID: J10, QTime Violation: 0.0
총 작업 수: 10, 완료된 작업 수: 2, 총 QTime 위반 시간: 40.786